# Week 3 assignment
Here is a short assignment to test your familiarity with the concepts sent. The assignments are analogous to the different stages of the final project we are going to implement.

This focuses on the preprocessing and transcription of the audio rather than focussing on the summarising part. Once we get the clean text, we can proceed to summarization in the following weeks. This is the most crucial part of the tool we are finally going to be building. 
### So gear up and get ready with your weapons

## Deliverables

- A short report covering your observations and findings across all tasks
- A Jupyter notebook (`.ipynb`) containing your full implementation and code
- Two folders:
  - `filtered_audio/` — containing audio files after bandpass filtering and spectral denoising
  - `vad_audio/` — containing audio files after VAD-based silence/non-speech removal




## Task 1:
An audio file has many properties such as duration, frequencies , amplitude, inactive durations, etc. It is important for us to get a clean audio file rather a one filled with background noises, distortions, etc. for our model to work better.  

* As we are already using a pretrained model for this task of transcription, we can focus on how to improve the results by finetuning the parameters.

* There are a few audio files given to you. Your first task is to carefully listen to them and manually categorise them into noisy,clean, and no voice.

* Create log-mel spectrograms which give an indication of what frequencies are present,required and what to remove. 

Finally note down the differences you observed between ones with noise and those without.

In [ ]:
import whisper

# Write your code here.

## Task 2: Signal Preprocessing — Bandpass Filtering and Spectral Denoising

Implement bandpass filtering and spectral denoising using modules such as `scipy.signal` and `librosa`, which are commonly used to process and analyze signals. Before jumping into code, it's worth understanding *why* these techniques help and what they're actually doing to the audio.

### Background: Why filter before transcribing?

Human speech energy is concentrated roughly between 80 Hz and 8 kHz. Anything outside that range (low rumble, high-frequency hiss, electrical hum) is very likely noise. A **bandpass filter** removes frequencies outside this range. **Spectral denoising** goes further — it estimates what the *noise itself* sounds like (often from a quiet stretch at the start of a clip) and subtracts that noise profile from the rest of the audio.

### Resources

To get comfortable with the underlying concepts before implementing:

- **Frequency domain & Fourier Transform intuition:** [3Blue1Brown — But what is the Fourier Transform?](https://www.youtube.com/watch?v=spUNpyF58BY) (visual, builds strong intuition)
- **Digital filters explained simply:** [Digital Filters in 5 minutes](https://www.youtube.com/watch?v=uNNNj9AZisM) or any short primer on low-pass/high-pass/band-pass filters
- **scipy.signal documentation:** [scipy.signal.butter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html) — for designing Butterworth filters
- **librosa documentation:** [librosa STFT](https://librosa.org/doc/main/generated/librosa.stft.html) — for converting audio to the frequency domain and back
- **Spectral subtraction (the technique behind our denoising):** [Spectral Subtraction for Speech Enhancement (short paper/explainer)](https://www.google.com/search?q=spectral+subtraction+speech+enhancement+explained) — skim for the core idea, don't worry about the math in depth

### What to implement

1. **Bandpass filter** — write a function that takes raw audio and removes frequencies outside the human speech range (~80 Hz–8 kHz) using `scipy.signal.butter` + `lfilter` (or `filtfilt`).
2. **Spectral denoising** — write a function that:
   - Converts audio to the frequency domain using `librosa.stft`
   - Estimates a noise profile from a quiet segment (e.g. the first 0.5s of the clip)
   - Subtracts that noise profile from the magnitude spectrum
   - Converts back to audio using `librosa.istft`
3. **Apply both** to your noisy clips from the earlier classification task, and save the processed audio as new `.wav` files.

### What to report

- Plot the log-mel spectrogram **before and after** filtering for one noisy clip (reuse your code from the earlier task). What changed visually?
- Did filtering ever *hurt* a clean clip? (Try it on one of your clean clips too — bandpass filters aren't free of risk.)

In [ ]:
### Write you code for task 2 here.

## Task 3 : Transcription

Properly transcribe using Whisper and observe how the transcription changes when parameters are changed. You can refer to the Whisper repository sent in the previous week's documentation and implement the code accordingly.

Pick at least 3-4 parameters to experiment with (e.g. `temperature`, `condition_on_previous_text`, `no_speech_threshold`, `initial_prompt`, beam size) and vary them one at a time so you can isolate their individual effect rather than changing everything at once.

Do this for **both** unprocessed and processed (filtered/denoised) audio files from the previous task, so you end up comparing four combinations per clip: raw audio with default params, raw audio with tuned params, processed audio with default params, and processed audio with tuned params.

Note down the transcripts for each run and briefly comment on what changed — did a parameter help more on noisy audio than clean audio? Did filtering reduce the need for aggressive parameter tuning, or did it make little difference?

In [ ]:
### Your code for task 3 goes down here.

## Task 4 (Bonus):

A speech file often contains pauses, interleaved silences, stammering, filler words ("um", "uh"), and other non-speech segments. Whisper generally handles these reasonably well on its own — but a customized solution tailored to your specific audio can perform better, especially for long lecture recordings where silent gaps or stammering can cause the model to hallucinate text that was never actually spoken.

Try implementing a **Voice Activity Detection (VAD)** step to identify and strip out these inactive/non-speech segments *before* feeding the audio into Whisper, rather than relying on Whisper to figure it out internally.

### Libraries you can explore

- **`webrtcvad`** — a lightweight, fast VAD originally built for WebRTC; works on short audio frames and is a good starting point since it's simple to set up and reason about.
- **`silero-vad`** — a more accurate, deep-learning-based VAD (available via `torch.hub`); tends to perform better on noisy or varied audio than `webrtcvad`, at the cost of being slightly heavier.


### What to do

1. Run VAD on a few of your audio clips and extract the timestamps where speech is actually present.
2. Strip out (or merge across) the non-speech segments, and feed only the speech portions into Whisper.
3. Compare the transcript against running Whisper on the raw, unprocessed audio.

### What to report

Does removing silences/non-speech segments reduce hallucinations or improve transcript quality, especially on longer clips? Did it cause any issues, like accidentally cutting off the start/end of a spoken word near a silence boundary?